<a href="https://colab.research.google.com/github/ghada-dahdoh/Applied-natural-language-processing/blob/main/Lab_3A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets scikit-learn pandas

In [ ]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
data = {
    "student_id": [101, 101, 102, 103, 103, 104, 105, 105],

    "text": [
        "أحتاج إلى تمديد موعد تسليم الواجب",
        "موعد المشروع قريب وأحتاج وقتًا إضافيًا",
        "لم تظهر لي درجة الاختبار في النظام",
        "أواجه مشكلة في تسجيل المقرر",
        "المقرر الذي أريده غير متاح للتسجيل",
        "القاعة الدراسية صغيرة ولا تكفي للطلاب",
        "لا أستطيع الدخول إلى نظام الجامعة",
        "الموقع الإلكتروني للجامعة لا يعمل"
    ],

    "topic": [
        "assignments",
        "assignments",
        "grades",
        "registration",
        "registration",
        "classrooms",
        "university_system",
        "university_system"
    ]
}

df = pd.DataFrame(data)

df

,student_id,text,topic
0,101,أحتاج إلى تمديد موعد تسليم الواجب,assignments
1,101,موعد المشروع قريب وأحتاج وقتًا إضافيًا,assignments
2,102,لم تظهر لي درجة الاختبار في النظام,grades
3,103,أواجه مشكلة في تسجيل المقرر,registration
4,103,المقرر الذي أريده غير متاح للتسجيل,registration
5,104,القاعة الدراسية صغيرة ولا تكفي للطلاب,classrooms
6,105,لا أستطيع الدخول إلى نظام الجامعة,university_system
7,105,الموقع الإلكتروني للجامعة لا يعمل,university_system


In [ ]:
topics = [
    "assignments",
    "grades",
    "registration",
    "classrooms",
    "university_system"
]

label_map = {
    topic: i for i, topic in enumerate(topics)
}

df["label"] = df["topic"].map(label_map)

df[["text", "topic", "label"]]

,text,topic,label
0,أحتاج إلى تمديد موعد تسليم الواجب,assignments,0
1,موعد المشروع قريب وأحتاج وقتًا إضافيًا,assignments,0
2,لم تظهر لي درجة الاختبار في النظام,grades,1
3,أواجه مشكلة في تسجيل المقرر,registration,2
4,المقرر الذي أريده غير متاح للتسجيل,registration,2
5,القاعة الدراسية صغيرة ولا تكفي للطلاب,classrooms,3
6,لا أستطيع الدخول إلى نظام الجامعة,university_system,4
7,الموقع الإلكتروني للجامعة لا يعمل,university_system,4


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df["student_id"])
)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train:")
print(train_df)

print("\nTest:")
print(test_df)

Train:
   student_id                                    text         topic  label
0         101       أحتاج إلى تمديد موعد تسليم الواجب   assignments      0
1         101  موعد المشروع قريب وأحتاج وقتًا إضافيًا   assignments      0
2         103             أواجه مشكلة في تسجيل المقرر  registration      2
3         103      المقرر الذي أريده غير متاح للتسجيل  registration      2
4         104   القاعة الدراسية صغيرة ولا تكفي للطلاب    classrooms      3

Test:
   student_id                                text              topic  label
0         102  لم تظهر لي درجة الاختبار في النظام             grades      1
1         105   لا أستطيع الدخول إلى نظام الجامعة  university_system      4
2         105   الموقع الإلكتروني للجامعة لا يعمل  university_system      4


In [ ]:
train_students = set(train_df["student_id"])
test_students = set(test_df["student_id"])

print(
    "Students in both sets:",
    train_students & test_students
)

Students in both sets: set()


In [ ]:
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer("أحتاج إلى تمديد موعد تسليم الواجب"))

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{'input_ids': [2, 3603, 4258, 1041, 2045, 19911, 5711, 10268, 12375, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
def tokenize_data(data):
    return tokenizer(
        data["text"].tolist(),
        truncation=True,
        padding=True
    )

train_tokens = tokenize_data(train_df)
test_tokens = tokenize_data(test_df)

print(train_tokens["input_ids"][0])

[2, 3603, 4258, 1041, 2045, 19911, 5711, 10268, 12375, 3]


In [ ]:
train_tokens["labels"] = train_df["label"].tolist()
test_tokens["labels"] = test_df["label"].tolist()

print(train_tokens["labels"])

[0, 0, 2, 2, 3]


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict(train_tokens)
test_dataset = Dataset.from_dict(test_tokens)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

print(model)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  439MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=2
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

print(trainer.evaluate())

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
No log,2.566863,12


{'eval_loss': 2.5668625831604004}
